# Bricky — Torso Embedding Training

End-to-end pipeline that produces the on-device torso identification model.
Run cells top-to-bottom. Total time on a free Colab T4: **~3-5 hours**.

**What you need before starting:**
1. Switch the runtime to a GPU: `Runtime → Change runtime type → T4 GPU`.
2. Nothing else. The notebook clones the repo, downloads the dataset, trains, and packages the artifacts.

**Output:** at the end you'll get a `torso_embeddings_bundle.zip` to download. Unzip into your local repo at `Bricky/Resources/TorsoEmbeddings/`, run `xcodegen generate`, rebuild — done.

## 1. Verify GPU

In [ ]:
!nvidia-smi || echo 'NO GPU — switch runtime to T4 before continuing'

## 2. Install dependencies

Colab already has torch + torchvision + Pillow. We add `coremltools` for the final export.

In [ ]:
!pip install -q coremltools==8.0 numpy pillow requests

## 3. Clone the repo

Public repo, no token needed. The notebook uses the training scripts in `Tools/torso-embeddings/` as-is.

In [ ]:
!git clone --depth 1 https://github.com/shribr/Bricky.git
%cd Bricky

## 4. Build the training dataset (~30 min)

Downloads ~14K torso reference renders from rebrickable's CDN. Each is
cropped to the torso band and padded to 224×224. Polite delay between
requests so we don't hammer the CDN.

In [ ]:
!python3 Tools/torso-embeddings/build-torso-dataset.py --sleep 0.02

Sanity check — count images and look at one.

In [ ]:
import glob
from PIL import Image
from IPython.display import display

imgs = sorted(glob.glob('Tools/torso-embeddings/data/figures/*.jpg'))
print(f'Have {len(imgs)} torso crops')
if imgs:
    display(Image.open(imgs[len(imgs) // 2]).resize((112, 112)))

## 5. Train the encoder (~2-4 hrs on T4)

Self-supervised contrastive (SimCLR-style). Two augmented views of each
torso crop are pulled together; views from different figures are pushed
apart. Defaults: 20 epochs, batch 128, ResNet18 backbone.

If Colab disconnects before this finishes you can lower `--epochs 12`
for a faster run with slightly lower accuracy.

In [ ]:
!python3 Tools/torso-embeddings/train-torso-encoder.py \
    --epochs 20 \
    --batch-size 128 \
    --num-workers 2

## 6. Embed the full catalog (~10 min)

Runs the trained encoder over every torso crop and writes the bundled
vector index that ships with the iOS app.

In [ ]:
!python3 Tools/torso-embeddings/embed-torso-catalog.py

## 7. Export to CoreML (~1 min)

Wraps the encoder for on-device inference and saves `TorsoEncoder.mlmodel`
into the same output directory as the embedding index.

In [ ]:
!python3 Tools/torso-embeddings/convert-torso-encoder-coreml.py

## 8. Package as a single zip and download

Bundles all three artifacts into `torso_embeddings_bundle.zip`.

In [ ]:
import os
import shutil
from pathlib import Path

out_dir = Path('Bricky/Resources/TorsoEmbeddings')
expected = ['torso_embeddings.bin', 'torso_embeddings_index.json', 'TorsoEncoder.mlmodel']
missing = [e for e in expected if not (out_dir / e).exists()]
if missing:
    raise RuntimeError(f'Pipeline did not produce: {missing}')

zip_path = '/content/torso_embeddings_bundle'
shutil.make_archive(zip_path, 'zip', out_dir)
size_mb = os.path.getsize(zip_path + '.zip') / (1024 * 1024)
print(f'Wrote {zip_path}.zip ({size_mb:.1f} MB)')
for e in expected:
    p = out_dir / e
    print(f'  {e}: {p.stat().st_size / (1024 * 1024):.2f} MB')

In [ ]:
from google.colab import files
files.download('/content/torso_embeddings_bundle.zip')

## 9. Install on your Mac

Run these commands locally after the download completes:

```bash
cd /path/to/Bricky
mkdir -p Bricky/Resources/TorsoEmbeddings
unzip -o ~/Downloads/torso_embeddings_bundle.zip -d Bricky/Resources/TorsoEmbeddings/
xcodegen generate
xcodebuild -project Bricky.xcodeproj -scheme Bricky build
```

On the next scan, the cascade automatically uses the trained model — the
runtime checks `TorsoEmbeddingService.shared.isAvailable` and lights up
Phase 1.5 once the artifacts are present.